In [9]:
import os
import sqlite3
import pandas as pd
import altair as alt
import kagglehub

# Download dataset
path = kagglehub.dataset_download("rtatman/188-million-us-wildfires")
print("Path to dataset files:", path)

# Locate SQLite database
db_path = os.path.join(path, "FPA_FOD_20170508.sqlite")

if not os.path.exists(db_path):
    raise FileNotFoundError(f"Database not found: {db_path}")

# Connect to database
conn = sqlite3.connect(db_path)

try:
    # Find available tables
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print("\nAvailable tables:")
    print(tables)

    if tables.empty:
        raise ValueError("No tables found in the SQLite database.")

    # Load the main wildfire records table
    table_name = " Fires "  # Placeholder; detected below
    candidates = tables["name"].tolist()

    # Prefer the table named 'Fires' (case-insensitive)
    fires_table = next(
        (name for name in candidates if name.strip().lower() == "fires"),
        candidates[0]
    )

    # Quote table name safely
    quoted_table = '"' + fires_table.replace('"', '""') + '"'

    # Load records into df
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table}", conn)

finally:
    conn.close()

# Inspect dataset
print("\nLoaded table:", fires_table)
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Path to dataset files: C:\Users\vgamb\.cache\kagglehub\datasets\rtatman\188-million-us-wildfires\versions\2

Available tables:
                                  name
0                      spatial_ref_sys
1                   spatialite_history
2                      sqlite_sequence
3                     geometry_columns
4                  spatial_ref_sys_aux
5               views_geometry_columns
6               virts_geometry_columns
7          geometry_columns_statistics
8    views_geometry_columns_statistics
9    virts_geometry_columns_statistics
10        geometry_columns_field_infos
11  views_geometry_columns_field_infos
12  virts_geometry_columns_field_infos
13               geometry_columns_time
14               geometry_columns_auth
15         views_geometry_columns_auth
16         virts_geometry_columns_auth
17                  sql_statements_log
18                        SpatialIndex
19                ElementaryGeometries
20                                 KNN
21             

,OBJECTID,FOD_ID,FPA_ID,SOURCE_SYSTEM_TYPE,SOURCE_SYSTEM,NWCG_REPORTING_AGENCY,NWCG_REPORTING_UNIT_ID,NWCG_REPORTING_UNIT_NAME,SOURCE_REPORTING_UNIT,SOURCE_REPORTING_UNIT_NAME,...,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,OWNER_CODE,OWNER_DESCR,STATE,COUNTY,FIPS_CODE,FIPS_NAME,Shape
0,1,1,FS-1418826,FED,FS-FIRESTAT,FS,USCAPNF,Plumas National Forest,0511,Plumas National Forest,...,A,40.036944,-121.005833,5.0,USFS,CA,63,063,Plumas,b'\x00\x01\xad\x10\x00\x00\xe8d\xc2\x92_@^\xc0...
1,2,2,FS-1418827,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,A,38.933056,-120.404444,5.0,USFS,CA,61,061,Placer,b'\x00\x01\xad\x10\x00\x00T\xb6\xeej\xe2\x19^\...
2,3,3,FS-1418835,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,A,38.984167,-120.735556,13.0,STATE OR PRIVATE,CA,17,017,El Dorado,b'\x00\x01\xad\x10\x00\x00\xd0\xa5\xa0W\x13/^\...
3,4,4,FS-1418845,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,A,38.559167,-119.913333,5.0,USFS,CA,3,003,Alpine,b'\x00\x01\xad\x10\x00\x00\x94\xac\xa3\rt\xfa]...
4,5,5,FS-1418847,FED,FS-FIRESTAT,FS,USCAENF,Eldorado National Forest,0503,Eldorado National Forest,...,A,38.559167,-119.933056,5.0,USFS,CA,3,003,Alpine,b'\x00\x01\xad\x10\x00\x00@\xe3\xaa.\xb7\xfb]\...


In [18]:
#df = df.drop(columns=["OBJECTID", "FIPS_CODE", "Shape", "SOURCE_SYSTEM_TYPE", "NWCG_REPORTING_AGENCY"])
df.head(5)
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1880465 entries, 0 to 1880464
Data columns (total 39 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   OBJECTID                    int64  
 1   FOD_ID                      int64  
 2   FPA_ID                      str    
 3   SOURCE_SYSTEM_TYPE          str    
 4   SOURCE_SYSTEM               str    
 5   NWCG_REPORTING_AGENCY       str    
 6   NWCG_REPORTING_UNIT_ID      str    
 7   NWCG_REPORTING_UNIT_NAME    str    
 8   SOURCE_REPORTING_UNIT       str    
 9   SOURCE_REPORTING_UNIT_NAME  str    
 10  LOCAL_FIRE_REPORT_ID        str    
 11  LOCAL_INCIDENT_ID           str    
 12  FIRE_CODE                   str    
 13  FIRE_NAME                   str    
 14  ICS_209_INCIDENT_NUMBER     str    
 15  ICS_209_NAME                str    
 16  MTBS_ID                     str    
 17  MTBS_FIRE_NAME              str    
 18  COMPLEX_NAME                str    
 19  FIRE_YEAR                   int6

In [14]:
# Aggregate data by year
yearly_fires = (
    df[df["FIRE_YEAR"].between(1992, 2015)]
    .groupby("FIRE_YEAR")
    .agg(
        Total_Acres_Burned=("FIRE_SIZE", "sum"),
        Number_of_Fires=("FOD_ID", "count")
    )
    .reset_index()
)
yearly_fires.to_csv(
    r"C:\Users\vgamb\Downloads\yearly_fires.csv",
    index=False
)
df.to_csv(
    r"C:\Users\vgamb\Downloads\df.csv",
    index=False
)
# Chart 1: Total acres burned
acres_chart = alt.Chart(yearly_fires).mark_line(point=True).encode(
    x=alt.X("FIRE_YEAR:O", title="Year"),
    y=alt.Y("Total_Acres_Burned:Q", title="Total Acres Burned"),
    tooltip=["FIRE_YEAR", "Total_Acres_Burned"]
).properties(
    title="Total Acres Burned by Year (1992–2015)",
    width=700,
    height=300
)
# Chart 2: Number of fires
fires_chart = alt.Chart(yearly_fires).mark_line(point=True).encode(
    x=alt.X("FIRE_YEAR:O", title="Year"),
    y=alt.Y("Number_of_Fires:Q", title="Number of Fires"),
    tooltip=["FIRE_YEAR", "Number_of_Fires"]
).properties(
    title="Number of Fires by Year (1992–2015)",
    width=700,
    height=300
)
acres_chart | fires_chart

alt.HConcatChart(...)

In [12]:
yearly_fires.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   FIRE_YEAR           24 non-null     int64  
 1   Total_Acres_Burned  24 non-null     float64
 2   Number_of_Fires     24 non-null     int64  
dtypes: float64(1), int64(2)
memory usage: 708.0 bytes


In [20]:
# ============================================================================
# Have wildfires become a bigger threat? Dumbbell chart by fire size class (D3),
# with filters to choose which two periods to compare.
# Paste this into a notebook cell.
# The D3 code below was generated by Claude (Anthropic); it is the one
# AI-generated visualization in this project.
# Needs: a DataFrame named `df` with one row per fire, and internet access
# (D3 is loaded from a CDN).
# ============================================================================
import html
import json
from pathlib import Path
import webbrowser

import numpy as np
from IPython.display import HTML, display

YEAR_COL  = "FIRE_YEAR"         # year of each fire
CLASS_COL = "FIRE_SIZE_CLASS"   # size class A-G (if missing, it is derived from SIZE_COL)
SIZE_COL  = "FIRE_SIZE"         # fire size in acres
EARLY     = (1992, 2003)        # period shown in blue when the chart first opens
LATE      = (2004, 2015)        # period shown in orange when the chart first opens
# (Viewers can change both periods with the filters above the chart.)

SAVE_AS         = "size_class_dumbbell.html"  # the chart is always saved here
OPEN_IN_BROWSER = True                        # open the saved file in your default browser
SHOW_INLINE     = True                        # also try to show it in the notebook
                                              # (VS Code may leave this blank)

# NWCG fire size classes, largest first (this is the row order, top to bottom).
CLASSES = [
    ("G", "5,000+ acres"),
    ("F", "1,000–4,999 acres"),
    ("E", "300–999 acres"),
    ("D", "100–299 acres"),
    ("C", "10–99 acres"),
    ("B", "0.26–9.9 acres"),
    ("A", "0–0.25 acres"),
]

if YEAR_COL not in df.columns:
    raise KeyError(f"Column '{YEAR_COL}' not found in df. Columns are: {list(df.columns)}")
if CLASS_COL not in df.columns and SIZE_COL not in df.columns:
    raise KeyError(f"Neither '{CLASS_COL}' nor '{SIZE_COL}' found in df. Columns are: {list(df.columns)}")

# Work on just the columns we need.
_cols = [YEAR_COL] + ([CLASS_COL] if CLASS_COL in df.columns else [SIZE_COL])
_d = df[_cols].dropna().copy()
if CLASS_COL in _d.columns:
    _d["cls"] = _d[CLASS_COL].astype(str).str.strip().str.upper()
else:
    _s = _d[SIZE_COL]
    _d["cls"] = np.select(
        [_s <= 0.25, _s < 10, _s < 100, _s < 300, _s < 1000, _s < 5000],
        list("ABCDEF"),
        default="G",
    )

_unknown = sorted(set(_d["cls"]) - {c for c, _ in CLASSES})
if _unknown:
    print("Note: ignoring unexpected size-class values:", _unknown)
_d = _d[_d["cls"].isin([c for c, _ in CLASSES])]

# Fires per year per size class. The chart does the averaging for whichever
# periods the viewer selects.
_by_year = (
    _d.groupby([YEAR_COL, "cls"]).size().reset_index(name="fires")
)
_by_year[YEAR_COL] = _by_year[YEAR_COL].astype(int)
fires_by_year = [
    {"year": int(r[YEAR_COL]), "cls": r["cls"], "fires": int(r["fires"])}
    for _, r in _by_year.iterrows()
]

# Quick table of the default periods so you can sanity-check the numbers.
def _total(code, lo, hi):
    m = (_by_year["cls"] == code) & _by_year[YEAR_COL].between(lo, hi)
    return int(_by_year.loc[m, "fires"].sum())

print(f"{'Class':<6}{'Fires ' + str(EARLY[0]) + '-' + str(EARLY[1]):>18}{'Fires ' + str(LATE[0]) + '-' + str(LATE[1]):>18}")
for _code, _label in CLASSES:
    print(f"{_code:<6}{_total(_code, *EARLY):>18,}{_total(_code, *LATE):>18,}")

PAGE = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Wildfire Size Classes</title>
<!--
  AI DISCLOSURE: The D3 code in this file was generated by Claude (Anthropic).
  It is the one AI-generated visualization in this project.
  Library: D3 v7 (https://d3js.org), loaded from jsDelivr.
  Data: aggregated from the notebook's DataFrame (embedded by the Python cell).
-->
<style>
  :root {
    --ink: #1f2933;
    --muted: #5f6b76;
    --grid: #e4e7eb;
    --link: #b8c0c9;
    --early: #3b6ea8;   /* first period  (blue)   */
    --late: #e8590c;    /* second period (orange) */
  }
  * { box-sizing: border-box; }
  body {
    margin: 0;
    background: #fff;
    color: var(--ink);
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    line-height: 1.5;
  }
  figure { max-width: 900px; margin: 0 auto; padding: 24px 16px 32px; position: relative; }
  h2 { font-size: 1.35rem; line-height: 1.25; margin: 0 0 6px; }
  .subtitle { margin: 0 0 14px; color: var(--muted); font-size: 0.95rem; max-width: 70ch; }

  .controls { display: flex; flex-direction: column; gap: 10px; margin: 0 0 14px; }
  .presets { display: flex; flex-wrap: wrap; gap: 8px; }
  .presets button {
    font: inherit; font-size: 0.85rem; padding: 5px 12px; cursor: pointer;
    border: 1px solid var(--link); border-radius: 999px; background: #fff; color: var(--ink);
  }
  .presets button:hover { border-color: var(--ink); }
  .presets button[aria-pressed="true"] { background: var(--ink); border-color: var(--ink); color: #fff; }
  .pickers { display: flex; flex-wrap: wrap; gap: 14px; }
  fieldset.picker {
    border: 1px solid var(--grid); border-radius: 8px; margin: 0;
    padding: 4px 12px 8px; font-size: 0.88rem;
  }
  fieldset.picker legend {
    padding: 0 6px; font-weight: 600; display: inline-flex; align-items: center; gap: 7px;
  }
  fieldset.picker legend i { width: 11px; height: 11px; border-radius: 50%; display: inline-block; }
  fieldset.picker label { margin-right: 8px; }
  fieldset.picker .total { margin-top: 6px; font-size: 0.85rem; color: var(--muted); }
  fieldset.picker .total b { color: var(--ink); font-weight: 600; }
  select {
    font: inherit; font-size: 0.88rem; padding: 3px 6px; color: var(--ink);
    border: 1px solid var(--link); border-radius: 6px; background: #fff;
  }

  .legend { display: flex; flex-wrap: wrap; gap: 18px; margin: 0 0 4px; font-size: 0.9rem; }
  .legend > span { display: inline-flex; align-items: center; gap: 7px; }
  .legend i { width: 12px; height: 12px; border-radius: 50%; display: inline-block; }
  svg { width: 100%; height: auto; display: block; }
  svg text { font-family: inherit; }
  .row-label { font-size: 13px; fill: var(--ink); }
  .gap-label { font-size: 12.5px; font-weight: 600; }
  .gap-head  { font-size: 11px; fill: var(--muted); }
  .axis-title { font-size: 11.5px; fill: var(--muted); }
  .axis text { font-size: 11.5px; fill: var(--muted); }
  .axis .tick line { stroke: var(--grid); }
  .axis .domain { display: none; }
  .row .link { stroke: var(--link); stroke-width: 3; stroke-linecap: round; }
  .row circle { stroke: #fff; stroke-width: 1.5; }
  .row.dim { opacity: 0.35; }
  #tooltip {
    position: absolute; pointer-events: none; opacity: 0;
    background: #1f2933; color: #fff; padding: 8px 10px; border-radius: 6px;
    font-size: 12.5px; line-height: 1.45; max-width: 280px;
    transition: opacity 0.12s; z-index: 5;
  }
  #tooltip b { display: block; margin-bottom: 2px; font-size: 13px; }
  .error { color: #b42318; font-size: 0.95rem; padding: 16px 0; }
  figcaption { margin-top: 14px; font-size: 0.8rem; color: var(--muted); max-width: 80ch; }
  figcaption p { margin: 0 0 6px; }
  sup { font-size: 0.7em; }
</style>
</head>
<body>

<figure>
  <h2>Are large wildfires becoming more common?<sup>†</sup></h2>
  <p class="subtitle">
    Average number of fires per year in each size class, for the two periods selected below.
    Rows run from the largest fires (top) to the smallest. The axis is logarithmic,
    so each gridline is ten times the one before it.
  </p>

  <div class="controls">
    <div class="presets" id="presets" role="group" aria-label="Quick comparisons"></div>
    <div class="pickers">
      <fieldset class="picker">
        <legend><i style="background:var(--early)"></i>Blue period</legend>
        <label>From <select id="a-from"></select></label>
        <label>to <select id="a-to"></select></label>
        <div class="total" id="a-total"></div>
      </fieldset>
      <fieldset class="picker">
        <legend><i style="background:var(--late)"></i>Orange period</legend>
        <label>From <select id="b-from"></select></label>
        <label>to <select id="b-to"></select></label>
        <div class="total" id="b-total"></div>
      </fieldset>
    </div>
  </div>

  <div class="legend" aria-hidden="true">
    <span><i style="background:var(--early)"></i><span id="legend-early"></span></span>
    <span><i style="background:var(--late)"></i><span id="legend-late"></span></span>
  </div>

  <div id="chart"></div>
  <div id="tooltip" role="status"></div>

  <figcaption>
    <p><strong>Data:</strong> USDA Forest Service Fire Program Analysis fire-occurrence database (Karen C. Short),
       accessed through Kaggle, “1.88 Million US Wildfires” (Rachael Tatman).
       Size classes follow the National Wildfire Coordinating Group standard. “Average per year” is the number of
       fires in the period divided by the number of years in the period; for a single year it is simply that
       year’s count. Totals include only fires with a recorded year and size class. Counts reflect reported
       fires in this dataset only, and reporting practices may have changed over time.</p>
    <p><sup>†</sup> <strong>AI disclosure:</strong> The code for this chart was generated by Claude (Anthropic).
       It is the one AI-generated visualization in this project; the other charts were built without AI assistance.</p>
  </figcaption>
</figure>

<script src="https://cdn.jsdelivr.net/npm/d3@7"></script>
<script>
  const FIRES    = __DATA__;       // [{year, cls, fires}]  one row per year and size class
  const CLASSES  = __CLASSES__;    // [{cls, label}]        largest class first
  const DEFAULTS = __DEFAULTS__;   // {a: [from, to], b: [from, to]}

  const fmtInt  = d3.format(",");
  const fmtRate = d3.format(",.1f");
  const fmtTick = d3.format(",");
  const periodLabel = (lo, hi) => lo === hi ? String(lo) : lo + "–" + hi;

  try {
    init();
  } catch (err) {
    document.getElementById("chart").innerHTML =
      '<p class="error">Could not draw the chart (' + err.message + ').</p>';
  }

  function init() {
    const years = Array.from(new Set(FIRES.map(d => +d.year))).sort((p, q) => p - q);
    if (!years.length) throw new Error("There are no fires in the data.");
    const counts = new Map(FIRES.map(d => [d.cls + "|" + d.year, +d.fires]));

    // Fires and fires-per-year for one size class over a range of years.
    function summarize(cls, lo, hi) {
      const ys = years.filter(y => y >= lo && y <= hi);
      const total = d3.sum(ys, y => counts.get(cls + "|" + y) || 0);
      return { total, perYear: ys.length ? total / ys.length : 0 };
    }

    // ---- Controls ---------------------------------------------------------
    const sel = {
      aFrom: d3.select("#a-from"), aTo: d3.select("#a-to"),
      bFrom: d3.select("#b-from"), bTo: d3.select("#b-to")
    };
    Object.values(sel).forEach(s =>
      s.selectAll("option").data(years).join("option").attr("value", d => d).text(d => d));

    const clampYear = y => years.includes(y) ? y : (y < years[0] ? years[0] : years[years.length - 1]);
    const n = years.length, half = Math.floor(n / 2);
    const PRESETS = [];
    if (n >= 4) {
      PRESETS.push({ name: "Early half vs. late half",
        a: [years[0], years[half - 1]], b: [years[half], years[n - 1]] });
    }
    if (n >= 2) {
      PRESETS.push({ name: "First year vs. last year",
        a: [years[0], years[0]], b: [years[n - 1], years[n - 1]] });
    }
    if (n >= 10) {
      PRESETS.push({ name: "First 5 vs. last 5 years",
        a: [years[0], years[4]], b: [years[n - 5], years[n - 1]] });
    }
    const presetBtns = d3.select("#presets").selectAll("button").data(PRESETS).join("button")
      .attr("type", "button").text(d => d.name)
      .on("click", (event, d) => setPeriods(d.a, d.b));

    // ---- Chart skeleton (drawn once, updated on every selection) ----------
    const width = 860, rowH = 44;
    const margin = { top: 60, right: 124, bottom: 24, left: 190 };
    const height = margin.top + margin.bottom + rowH * CLASSES.length;

    const svg = d3.select("#chart").append("svg")
      .attr("viewBox", [0, 0, width, height])
      .attr("role", "img")
      .attr("aria-label",
        "Dumbbell chart comparing the average number of wildfires per year in each fire size class " +
        "between two selected periods, on a logarithmic axis.");

    const x = d3.scaleLog().range([margin.left, width - margin.right]);
    const y = d3.scaleBand().domain(CLASSES.map(d => d.cls))
      .range([margin.top, height - margin.bottom]).padding(0.3);

    svg.append("text").attr("class", "axis-title")
      .attr("x", margin.left).attr("y", 12).text("Average fires per year (log scale)");
    svg.append("text").attr("class", "gap-head")
      .attr("x", width - 8).attr("y", 12).attr("text-anchor", "end")
      .text("Change, blue → orange");
    const axisG = svg.append("g").attr("class", "axis")
      .attr("transform", `translate(0,${margin.top - 8})`);

    const data = CLASSES.map(c => ({ ...c, A: null, B: null, early: 0, late: 0 }));
    const tooltip = d3.select("#tooltip");
    const figure = document.querySelector("figure");

    const rows = svg.selectAll("g.row").data(data).join("g")
      .attr("class", "row")
      .attr("transform", d => `translate(0,${y(d.cls) + y.bandwidth() / 2})`);

    rows.append("line").attr("class", "link").attr("y1", 0).attr("y2", 0);
    rows.append("circle").attr("class", "c-early").attr("r", 7).attr("fill", "var(--early)");
    rows.append("circle").attr("class", "c-late").attr("r", 7).attr("fill", "var(--late)");

    const labels = rows.append("text").attr("class", "row-label")
      .attr("x", margin.left - 14).attr("y", 0).attr("dy", "0.35em").attr("text-anchor", "end");
    labels.append("tspan").attr("font-weight", 700).text(d => "Class " + d.cls);
    labels.append("tspan").text(d => "  ·  " + d.label);

    const gapLabels = rows.append("text").attr("class", "gap-label")
      .attr("x", width - 8).attr("y", 0).attr("dy", "0.35em").attr("text-anchor", "end");

    const pctChange = d => d.early > 0 ? (d.late - d.early) / d.early * 100 : null;
    // Values that round to zero show as "0%" with no sign.
    const fmtChange = (p, digits) => {
      if (p === null) return "n/a";
      const r = +Math.abs(p).toFixed(digits);
      if (r === 0) return (0).toFixed(digits) + "%";
      return (p > 0 ? "+" : "−") + r.toFixed(digits) + "%";
    };

    let current = { a: DEFAULTS.a, b: DEFAULTS.b };
    const describe = (lo, hi, s) => lo === hi
      ? `${lo}: ${fmtInt(s.total)} fires`
      : `${periodLabel(lo, hi)}: ${fmtInt(s.total)} fires (${fmtRate(s.perYear)} per year)`;

    rows.append("rect")
      .attr("x", 0).attr("width", width).attr("y", -rowH / 2).attr("height", rowH)
      .attr("fill", "transparent")
      .on("pointerenter", function (event, d) {
        rows.classed("dim", r => r !== d);
        tooltip.style("opacity", 1).html(
          `<b>Class ${d.cls} · ${d.label}</b>` +
          `${describe(current.a[0], current.a[1], d.A)}<br>` +
          `${describe(current.b[0], current.b[1], d.B)}<br>` +
          `Change: ${fmtChange(pctChange(d), 1)}`
        );
      })
      .on("pointermove", function (event) {
        const box = figure.getBoundingClientRect();
        let left = event.clientX - box.left + 14;
        const maxLeft = box.width - 290;
        if (left > maxLeft) left = maxLeft;
        tooltip.style("left", left + "px").style("top", (event.clientY - box.top + 14) + "px");
      })
      .on("pointerleave", function () {
        rows.classed("dim", false);
        tooltip.style("opacity", 0);
      });

    // ---- Selection handling ------------------------------------------------
    const readYear = s => +s.property("value");
    const same = (p, q) => p[0] === q[0] && p[1] === q[1];

    function setPeriods(a, b) {
      sel.aFrom.property("value", clampYear(a[0])); sel.aTo.property("value", clampYear(a[1]));
      sel.bFrom.property("value", clampYear(b[0])); sel.bTo.property("value", clampYear(b[1]));
      update(true);
    }

    let firstDraw = true;
    function update(animate) {
      // Read the dropdowns; if "from" is after "to", swap them.
      let a = [readYear(sel.aFrom), readYear(sel.aTo)];
      let b = [readYear(sel.bFrom), readYear(sel.bTo)];
      if (a[0] > a[1]) a = [a[1], a[0]];
      if (b[0] > b[1]) b = [b[1], b[0]];
      sel.aFrom.property("value", a[0]); sel.aTo.property("value", a[1]);
      sel.bFrom.property("value", b[0]); sel.bTo.property("value", b[1]);
      current = { a, b };

      data.forEach(d => {
        d.A = summarize(d.cls, a[0], a[1]);
        d.B = summarize(d.cls, b[0], b[1]);
        d.early = d.A.perYear;
        d.late = d.B.perYear;
      });

      document.getElementById("legend-early").textContent =
        a[0] === a[1] ? `Fires in ${a[0]}` : `Average per year, ${periodLabel(a[0], a[1])}`;
      document.getElementById("legend-late").textContent =
        b[0] === b[1] ? `Fires in ${b[0]}` : `Average per year, ${periodLabel(b[0], b[1])}`;
      presetBtns.attr("aria-pressed", d => same(d.a, a) && same(d.b, b));

      // Total fires reported (all size classes) under each period's dropdowns.
      const showTotal = (id, range, key) => {
        const total = d3.sum(data, d => d[key].total);
        const nYears = years.filter(yr => yr >= range[0] && yr <= range[1]).length;
        document.getElementById(id).innerHTML =
          `Total fires reported: <b>${fmtInt(total)}</b>` +
          (nYears > 1 ? ` (${fmtInt(Math.round(total / nYears))} per year)` : "");
      };
      showTotal("a-total", a, "A");
      showTotal("b-total", b, "B");

      // Log scale: round the domain out to powers of ten for the current selection.
      const positives = data.flatMap(d => [d.early, d.late]).filter(v => v > 0);
      if (!positives.length) throw new Error("There are no fires in the selected periods.");
      const e0 = Math.floor(Math.log10(d3.min(positives)));
      let e1 = Math.ceil(Math.log10(d3.max(positives)));
      if (e1 <= e0) e1 = e0 + 1;
      const xMin = Math.pow(10, e0), xMax = Math.pow(10, e1);
      x.domain([xMin, xMax]);
      const pos = v => x(Math.max(v, xMin));          // zero counts sit at the left edge
      const ticks = d3.range(e0, e1 + 1).map(e => Math.pow(10, e));

      const t = d3.transition().duration(animate && !firstDraw ? 450 : 0);
      axisG.transition(t).call(
        d3.axisTop(x).tickValues(ticks).tickFormat(v => fmtTick(v))
          .tickSize(-(height - margin.top - margin.bottom + 8)).tickPadding(8));

      rows.select("line.link").transition(t)
        .attr("x1", d => pos(Math.min(d.early, d.late)))
        .attr("x2", d => pos(Math.max(d.early, d.late)));
      rows.select("circle.c-early").transition(t).attr("cx", d => pos(d.early));
      rows.select("circle.c-late").transition(t).attr("cx", d => pos(d.late));

      gapLabels
        .attr("fill", d => {
          const p = pctChange(d);
          if (p === null || Math.round(p) === 0) return "var(--muted)";
          return p > 0 ? "var(--late)" : "var(--early)";
        })
        .text(d => fmtChange(pctChange(d), 0));

      firstDraw = false;
    }

    [sel.aFrom, sel.aTo, sel.bFrom, sel.bTo].forEach(s => s.on("change", () => update(true)));

    // Start with the default periods.
    const d0 = [clampYear(DEFAULTS.a[0]), clampYear(DEFAULTS.a[1])];
    const d1 = [clampYear(DEFAULTS.b[0]), clampYear(DEFAULTS.b[1])];
    setPeriods(d0, d1);
  }
</script>
</body>
</html>
'''

_payload  = json.dumps(fires_by_year).replace("</", "<\\/")
_classes  = json.dumps([{"cls": c, "label": l} for c, l in CLASSES])
_defaults = json.dumps({"a": list(EARLY), "b": list(LATE)})
_page = (PAGE.replace("__DATA__", _payload)
             .replace("__CLASSES__", _classes)
             .replace("__DEFAULTS__", _defaults))

_out = Path(SAVE_AS).resolve()
_out.write_text(_page, encoding="utf-8")
print(f"\nChart saved to: {_out}")

if OPEN_IN_BROWSER:
    webbrowser.open(_out.as_uri())

if SHOW_INLINE:
    _height = len(CLASSES) * 44 + 84 + 320 + 130   # rows + title, filters, legend and caption text
    display(HTML(
        f'<iframe srcdoc="{html.escape(_page, quote=True)}" '
        f'width="100%" height="{_height}" style="border:0"></iframe>'
    ))

Class    Fires 1992-2003   Fires 2004-2015
G                  1,486             2,287
F                  3,411             4,375
E                  6,265             7,842
D                 12,569            15,858
C                104,989           115,088
B                454,531           484,845
A                315,817           351,102

Chart saved to: C:\Users\vgamb\AppData\Local\Programs\Microsoft VS Code\size_class_dumbbell.html


C:\Users\vgamb\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\IPython\core\display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
